In [1]:
import pandas as pd
import numpy as  np


In [3]:
df = pd.read_csv(r'/content/online_retail_II.csv', encoding='ISO-8859-1')

In [8]:
from collections import Counter
from itertools import combinations

# 3. Remove cancelled invoices
df = df[~df["Invoice"].astype(str).str.startswith("C")]


# 4. Keep only positive quantities
df = df[df["Quantity"] > 0]


# 5. Remove missing values
df = df.dropna()


# 6. Remove duplicate rows
df = df.drop_duplicates()


# 7. Check dataset
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique invoices:", df["Invoice"].nunique())
print("Unique products:", df["StockCode"].nunique())


# ============================================
# CREATE INVOICE-PRODUCT BASKET
# ============================================

# 8. Group by Invoice and Product
basket = df.groupby(["Invoice", "StockCode"])["Quantity"].sum()

print(basket)
# 9. Convert grouped data into a table
basket = basket.unstack(fill_value=0)
print(basket)

# 10. Convert quantities into 1/0
# 1 = product purchased
# 0 = product not purchased
basket = basket.map(lambda x: 1 if x > 0 else 0)
print(basket)

# ============================================
# FIND PRODUCTS PURCHASED TOGETHER
# ============================================

# 11. Create product pairs for each invoice
pair_counts = Counter()

for invoice in basket.index:

    products = basket.loc[invoice]
    purchased_products = products[products == 1].index.tolist()

    for pair in combinations(purchased_products, 2):
      pair_counts[pair] += 1


# 12. Convert pairs into DataFrame
pairs = pd.DataFrame(
    pair_counts.items(),
    columns=["Product_Pair", "Times_Bought_Together"]
)


# 13. Sort by most frequently purchased together
pairs = pairs.sort_values(
    "Times_Bought_Together",
    ascending=False
)


# 14. Display top 20 product pairs
print(pairs.head(20))

Rows: 46929
Columns: 8
Unique invoices: 2399
Unique products: 2629
Invoice  StockCode
536365   21730         6
         22752         2
         71053         6
         84029E        6
         84029G        6
                      ..
542791   47566B        3
         84313B        3
         84508A        1
         85099B       20
         85099C       10
Name: Quantity, Length: 46348, dtype: int64
StockCode  10002  10120  10123C  10124A  10124G  10125  10133  10135  11001  \
Invoice                                                                       
536365         0      0       0       0       0      0      0      0      0   
536366         0      0       0       0       0      0      0      0      0   
536367         0      0       0       0       0      0      0      0      0   
536368         0      0       0       0       0      0      0      0      0   
536369         0      0       0       0       0      0      0      0      0   
...          ...    ...     ...     ...   

In [9]:

basket_data = df[
    ["Invoice", "StockCode"]
].drop_duplicates()


print("\nBasket Data:")
print(basket_data.head())


# ============================================
# 9. CREATE PRODUCT PAIRS
# ============================================

pairs = basket_data.merge(
    basket_data,
    on="Invoice"
)


print("\nAfter Self Merge:")
print(pairs.head(20))


# ============================================
# 10. REMOVE SAME PRODUCT
#     AND REVERSE DUPLICATES
# ============================================

pairs = pairs[
    pairs["StockCode_x"] < pairs["StockCode_y"]
]


print("\nUnique Product Pairs:")
print(pairs.head(20))


# ============================================
# 11. COUNT HOW MANY TIMES
#     EACH PAIR WAS PURCHASED
# ============================================

pair_counts = (
    pairs
    .groupby(
        ["StockCode_x", "StockCode_y"]
    )
    .size()
    .reset_index(
        name="Times_Bought_Together"
    )
)


# ============================================
# 12. SORT BY MOST FREQUENT PAIRS
# ============================================

pair_counts = pair_counts.sort_values(
    "Times_Bought_Together",
    ascending=False
)


# ============================================
# 13. DISPLAY TOP 20 PRODUCT PAIRS
# ============================================

print("\nTop 20 Products Purchased Together:")

print(
    pair_counts.head(20)
)


Basket Data:
  Invoice StockCode
0  536365    85123A
1  536365     71053
2  536365    84406B
3  536365    84029G
4  536365    84029E

After Self Merge:
   Invoice StockCode_x StockCode_y
0   536365      85123A      85123A
1   536365      85123A       71053
2   536365      85123A      84406B
3   536365      85123A      84029G
4   536365      85123A      84029E
5   536365      85123A       22752
6   536365      85123A       21730
7   536365       71053      85123A
8   536365       71053       71053
9   536365       71053      84406B
10  536365       71053      84029G
11  536365       71053      84029E
12  536365       71053       22752
13  536365       71053       21730
14  536365      84406B      85123A
15  536365      84406B       71053
16  536365      84406B      84406B
17  536365      84406B      84029G
18  536365      84406B      84029E
19  536365      84406B       22752

Unique Product Pairs:
   Invoice StockCode_x StockCode_y
7   536365       71053      85123A
9   536365       71